In [1]:
import re
from datetime import datetime

# ---------- SETTINGS (change these to your own) ----------
CURRENCY = "RWF"
MENU = {"rice": 1000, "beans": 500, "chapati": 300, "chips": 800, "juice": 500}
OPEN_HOURS = "Monday to Friday, 7:00 to 16:00"
LOCAL_GREETINGS = ["muraho", "mwaramutse", "mwiriwe"]  # example local words: replace with yours
MAX_MISSES = 2

# ---------- STATE ----------
state = {"pending": None, "last_order": None, "misses": 0}
log = []  # messages the bot did not understand

# ---------- TEXT NORMALIZER ----------
def normalize(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)   # remove punctuation
    text = re.sub(r"\s+", " ", text)       # collapse spaces
    return text

# ---------- HELPERS ----------
NUMBER_WORDS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}

def find_items(text):
    items = {}
    for name in MENU:
        m = re.search(r"(?:(\d+|one|two|three|four|five)\s+)?\b" + name + r"s?\b", text)
        if m:
            q = m.group(1)
            qty = int(q) if q and q.isdigit() else NUMBER_WORDS.get(q, 1)
            items[name] = qty
    return items

def total_cost(items):
    return sum(MENU[n] * q for n, q in items.items())

# ---------- HANDLERS ----------
def greet(text):
    return "Hello! Welcome to the school canteen bot. You can ask for the menu, prices, opening hours, or place an order."

def show_menu(text):
    lines = [f"- {n.title()}: {p} {CURRENCY}" for n, p in MENU.items()]
    return "Today's menu:\n" + "\n".join(lines)

def show_price(text):
    items = find_items(text)
    if not items:
        return "Which item? I can tell you the price of: " + ", ".join(MENU)
    return "\n".join(f"{n.title()} costs {MENU[n]} {CURRENCY}." for n in items)

def take_order(text):
    items = find_items(text)
    if not items:
        return "What would you like to order? We have: " + ", ".join(MENU)
    state["pending"] = items
    summary = ", ".join(f"{q} x {n}" for n, q in items.items())
    return f"Your order: {summary}. Total {total_cost(items)} {CURRENCY}. Confirm? (yes/no)"

def opening_hours(text):
    return f"The canteen is open {OPEN_HOURS}."

def order_status(text):
    if state["last_order"]:
        return "Your order is being prepared. It will be ready in about 10 minutes."
    return "You have no order yet. Say 'I want 2 chapati' to start one."

def human_help(text):
    return "Okay, I'm connecting you to a canteen staff member. Please wait at the counter."

def exit_bot(text):
    return "Goodbye! Enjoy your meal."

# ---------- RULE TABLE (tried in order) ----------
RULES = [
    ("greeting", r"\b(hi|hello|hey|good (morning|afternoon|evening)|" + "|".join(LOCAL_GREETINGS) + r")\b", greet),
    ("status",   r"\b(where is my order|order status|status|is it ready)\b", order_status),
    ("price",    r"\b(how much|price|cost)\b", show_price),
    ("menu",     r"\b(menu|sell|available|what do you have|food)\b", show_menu),
    ("order",    r"\b(order|want|buy|give me|get me|i ll take)\b", take_order),
    ("hours",    r"\b(open|close|closing|hours)\b", opening_hours),
    ("human",    r"\b(person|human|staff|agent|help)\b", human_help),
]

EXIT_PATTERN = r"\b(bye|goodbye|quit|exit|stop)\b"

# ---------- FALLBACK ----------
def fallback(original):
    state["misses"] += 1
    log.append(f"{datetime.now():%H:%M:%S} | {original}")
    if state["misses"] >= MAX_MISSES:
        return "Sorry, I keep missing that. Would you like to talk to a person? Say 'staff'."
    return "Sorry, I didn't understand. I can show the menu, give prices, take an order, or tell you opening hours."

# ---------- ROUTER ----------
def reply(user_text):
    text = normalize(user_text)

    # 1. exit works at ANY point
    if re.search(EXIT_PATTERN, text):
        state["pending"] = None
        return exit_bot(text)

    # 2. confirmation step
    if state["pending"]:
        if re.search(r"\b(yes|yeah|yep|ok|okay|confirm)\b", text):
            state["last_order"] = state["pending"]
            state["pending"] = None
            return "Order confirmed! Thank you."
        if re.search(r"\b(no|nope|cancel)\b", text):
            state["pending"] = None
            return "Order cancelled. Anything else?"
        return "Please answer yes or no to confirm your order."

    # 3. rule table
    for name, pattern, handler in RULES:
        if re.search(pattern, text):
            state["misses"] = 0
            return handler(text)

    # 4. fallback
    return fallback(user_text)

# ---------- CHAT LOOP ----------
def chat():
    print("Canteen Bot: Hi! Type 'bye' any time to leave.")
    while True:
        user = input("You: ")
        answer = reply(user)
        print("Canteen Bot:", answer)
        if re.search(EXIT_PATTERN, normalize(user)):
            break

In [2]:
print(reply("Hello!!"))
print(reply("what do you sell?"))
print(reply("I want 2 chapatis"))
print(reply("yes"))
print(reply("bye"))

Hello! Welcome to the school canteen bot. You can ask for the menu, prices, opening hours, or place an order.
Today's menu:
- Rice: 1000 RWF
- Beans: 500 RWF
- Chapati: 300 RWF
- Chips: 800 RWF
- Juice: 500 RWF
Your order: 2 x chapati. Total 600 RWF. Confirm? (yes/no)
Order confirmed! Thank you.
Goodbye! Enjoy your meal.


In [13]:
test_plan = """
TEST PLAN (written before running)

1  | Hello!!                       | Greeting message
2  | muraho                        | Greeting message (local greeting)
3  | what do you sell?             | Menu list with prices
4  | how much is chapati           | Chapati costs 300 RWF
5  | I want 2 chapatis and 1 juice | Order summary, total 1100 RWF, asks to confirm
6  | (after test 5) yes            | Order confirmed
7  | when do you open?             | Opening hours
8  | where is my order             | Says order is being prepared
9  | asdfgh                        | Fallback message (not understood)
10 | bye                           | Goodbye message
"""
print(test_plan)


TEST PLAN (written before running)

1  | Hello!!                       | Greeting message
2  | muraho                        | Greeting message (local greeting)
3  | what do you sell?             | Menu list with prices
4  | how much is chapati           | Chapati costs 300 RWF
5  | I want 2 chapatis and 1 juice | Order summary, total 1100 RWF, asks to confirm
6  | (after test 5) yes            | Order confirmed
7  | when do you open?             | Opening hours
8  | where is my order             | Says order is being prepared
9  | asdfgh                        | Fallback message (not understood)
10 | bye                           | Goodbye message



In [14]:
# reset the bot before testing
state["pending"] = None
state["last_order"] = None
state["misses"] = 0

tests = [
    "Hello!!",
    "muraho",
    "what do you sell?",
    "how much is chapati",
    "I want 2 chapatis and 1 juice",
    "yes",
    "when do you open?",
    "where is my order",
    "asdfgh",
    "bye",
]

for i, t in enumerate(tests, 1):
    print(f"Test {i}: {t}")
    print("Bot:", reply(t))
    print("-" * 40)

Test 1: Hello!!
Bot: Hello! Welcome to the school canteen bot. You can ask for the menu, prices, opening hours, or place an order.
----------------------------------------
Test 2: muraho
Bot: Hello! Welcome to the school canteen bot. You can ask for the menu, prices, opening hours, or place an order.
----------------------------------------
Test 3: what do you sell?
Bot: Today's menu:
- Rice: 1000 RWF
- Beans: 500 RWF
- Chapati: 300 RWF
- Chips: 800 RWF
- Juice: 500 RWF
----------------------------------------
Test 4: how much is chapati
Bot: Chapati costs 300 RWF.
----------------------------------------
Test 5: I want 2 chapatis and 1 juice
Bot: Your order: 2 x chapati, 1 x juice. Total 1100 RWF. Confirm? (yes/no)
----------------------------------------
Test 6: yes
Bot: Order confirmed! Thank you.
----------------------------------------
Test 7: when do you open?
Bot: The canteen is open Monday to Friday, 7:00 to 16:00.
----------------------------------------
Test 8: where is my or

In [15]:
test_plan_round_2 = """
ROUND 2 TEST PLAN (written before running)

11 | hi, how much is rice            | Rice costs 1000 RWF.
12 | Hello, I want 2 beans           | Order summary: 2 x beans, total 1000 RWF, asks to confirm
13 | what time do you stop selling?  | Opening hours
14 | can you help me order chips     | Order summary: 1 x chips, total 800 RWF, asks to confirm
15 | muraho, ndashaka umuceri        | Understands the local phrasing (greeting or order help)
"""
print(test_plan_round_2)


ROUND 2 TEST PLAN (written before running)

11 | hi, how much is rice            | Rice costs 1000 RWF.
12 | Hello, I want 2 beans           | Order summary: 2 x beans, total 1000 RWF, asks to confirm
13 | what time do you stop selling?  | Opening hours
14 | can you help me order chips     | Order summary: 1 x chips, total 800 RWF, asks to confirm
15 | muraho, ndashaka umuceri        | Understands the local phrasing (greeting or order help)



In [16]:
state["pending"] = None
state["last_order"] = None
state["misses"] = 0

tests2 = [
    "hi, how much is rice",
    "Hello, I want 2 beans",
    "what time do you stop selling?",
    "can you help me order chips",
    "muraho, ndashaka umuceri",
]

for i, t in enumerate(tests2, 11):
    print(f"Test {i}: {t}")
    print("Bot:", reply(t))
    print("-" * 40)

Test 11: hi, how much is rice
Bot: Hello! Welcome to the school canteen bot. You can ask for the menu, prices, opening hours, or place an order.
----------------------------------------
Test 12: Hello, I want 2 beans
Bot: Hello! Welcome to the school canteen bot. You can ask for the menu, prices, opening hours, or place an order.
----------------------------------------
Test 13: what time do you stop selling?
Bot: Goodbye! Enjoy your meal.
----------------------------------------
Test 14: can you help me order chips
Bot: Your order: 1 x chips. Total 800 RWF. Confirm? (yes/no)
----------------------------------------
Test 15: muraho, ndashaka umuceri
Bot: Please answer yes or no to confirm your order.
----------------------------------------


In [17]:
# ---------- FIX 1: "stop" only exits when it is the whole message ----------
EXIT_PATTERN = r"\b(bye|goodbye|quit|exit)\b|^stop$"

# ---------- FIX 2: understand local words ----------
LOCAL_WORDS = {"umuceri": "rice", "ibishyimbo": "beans"}

def find_items(text):
    for local, english in LOCAL_WORDS.items():
        text = text.replace(local, english)
    items = {}
    for name in MENU:
        m = re.search(r"(?:(\d+|one|two|three|four|five)\s+)?\b" + name + r"s?\b", text)
        if m:
            q = m.group(1)
            qty = int(q) if q and q.isdigit() else NUMBER_WORDS.get(q, 1)
            items[name] = qty
    return items

# ---------- FIX 3: greeting moved to the END of the rule table ----------
RULES = [
    ("status",   r"\b(where is my order|order status|status|is it ready)\b", order_status),
    ("price",    r"\b(how much|price|cost)\b", show_price),
    ("menu",     r"\b(menu|sell|available|what do you have|food)\b", show_menu),
    ("order",    r"\b(order|want|buy|give me|get me|i ll take|ndashaka)\b", take_order),
    ("hours",    r"\b(open|close|closing|hours|time)\b", opening_hours),
    ("human",    r"\b(person|human|staff|agent|help)\b", human_help),
    ("greeting", r"\b(hi|hello|hey|good (morning|afternoon|evening)|" + "|".join(LOCAL_GREETINGS) + r")\b", greet),
]

In [18]:
for i, t in enumerate(tests2, 11):
    state["pending"] = None
    state["misses"] = 0
    print(f"Test {i}: {t}")
    print("Bot:", reply(t))
    print("-" * 40)

Test 11: hi, how much is rice
Bot: Rice costs 1000 RWF.
----------------------------------------
Test 12: Hello, I want 2 beans
Bot: Your order: 2 x beans. Total 1000 RWF. Confirm? (yes/no)
----------------------------------------
Test 13: what time do you stop selling?
Bot: The canteen is open Monday to Friday, 7:00 to 16:00.
----------------------------------------
Test 14: can you help me order chips
Bot: Your order: 1 x chips. Total 800 RWF. Confirm? (yes/no)
----------------------------------------
Test 15: muraho, ndashaka umuceri
Bot: Your order: 1 x rice. Total 1000 RWF. Confirm? (yes/no)
----------------------------------------


In [19]:
state["pending"] = None
state["last_order"] = None
state["misses"] = 0

for i, t in enumerate(tests, 1):
    print(f"Test {i}: {t}")
    print("Bot:", reply(t))
    print("-" * 40)

Test 1: Hello!!
Bot: Hello! Welcome to the school canteen bot. You can ask for the menu, prices, opening hours, or place an order.
----------------------------------------
Test 2: muraho
Bot: Hello! Welcome to the school canteen bot. You can ask for the menu, prices, opening hours, or place an order.
----------------------------------------
Test 3: what do you sell?
Bot: Today's menu:
- Rice: 1000 RWF
- Beans: 500 RWF
- Chapati: 300 RWF
- Chips: 800 RWF
- Juice: 500 RWF
----------------------------------------
Test 4: how much is chapati
Bot: Chapati costs 300 RWF.
----------------------------------------
Test 5: I want 2 chapatis and 1 juice
Bot: Your order: 2 x chapati, 1 x juice. Total 1100 RWF. Confirm? (yes/no)
----------------------------------------
Test 6: yes
Bot: Order confirmed! Thank you.
----------------------------------------
Test 7: when do you open?
Bot: The canteen is open Monday to Friday, 7:00 to 16:00.
----------------------------------------
Test 8: where is my or